<a href="https://colab.research.google.com/github/lzxatdk-tech/NLP_project/blob/main/reliable_multilingual_question_answering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Environment Setup

## Mount data folder

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
folder_path = '/content/drive/MyDrive/Shared_NLP_Project'

os.chdir(folder_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Download or load dataset

In [4]:
import os
from datasets import load_dataset, load_from_disk

TARGET_DIR = "./tydi_xor_rc"

if not os.path.exists(TARGET_DIR):
    print(f"Downloading dataset to {TARGET_DIR}...")
    dataset = load_dataset("coastalcph/tydi_xor_rc")
    dataset.save_to_disk(TARGET_DIR)
    print("Download and save complete.")
else:
    print(f"Found existing dataset at {TARGET_DIR}. Loading from disk...")
    dataset = load_from_disk(TARGET_DIR)
    print("Loading completes.")

df_train = dataset["train"].to_pandas()
df_val = dataset["validation"].to_pandas()

Found existing dataset at ./tydi_xor_rc. Loading from disk...
Loading completes.


In [10]:
# tokenizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

In [33]:
tokenizer.tokenize("The exact number of Arab casualties is unknown. One estimate places the Arab death toll at 7,000,")

['▁The',
 '▁exact',
 '▁number',
 '▁of',
 '▁Arab',
 '▁casual',
 'ties',
 '▁is',
 '▁un',
 'know',
 'n',
 '.',
 '▁One',
 '▁estima',
 'te',
 '▁places',
 '▁the',
 '▁Arab',
 '▁death',
 '▁toll',
 '▁at',
 '▁',
 '7,000',
 ',']

## BIO Labels

In [11]:
def is_answer_valid(answer, answer_start, context):
  return not (not answer or answer_start is None or answer_start < 0)

def bio_labelller(context, answer, answer_start):
  encoding = tokenizer(
      context,
      add_special_tokens=False,
      return_offsets_mapping=True,
  )

  tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"])
  offsets = encoding["offset_mapping"]

  labels = ["O"] * len(tokens)

  if not is_answer_valid(answer, answer_start, context):
      return tokens, offsets, labels

  answer_end = answer_start + len(answer)

  for token_index, (token_start, token_end) in enumerate(offsets):
      if token_start <= answer_start and token_end > answer_start:
          labels[token_index] = "B"
      elif token_start < answer_end and token_start > answer_start:
          labels[token_index] = "I"

  return tokens, offsets, labels

def bio_labelller_row(row):
  context = row["context"]
  answer = row["answer"]
  answer_start = row["answer_start"]

  tokens, offsets, labels = bio_labelller(context, answer, answer_start)

  return labels


In [12]:
df_train["sequence_labels"] = df_train.apply(
    bio_labelller_row,
    axis=1,
)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (648 > 512). Running this sequence through the model will result in indexing errors


In [13]:
df_train.head()

,question,context,lang,answerable,answer_start,answer,answer_inlang,sequence_labels
0,উইকিলিকস কত সালে সর্বপ্রথম ইন্টারনেটে প্রথম তথ...,WikiLeaks () is an international non-profit or...,bn,True,182,2006,None,"[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
1,দ্বিতীয় বিশ্বযুদ্ধে কোন দেশ পরাজিত হয় ?,The war in Europe concluded with an invasion o...,bn,True,48,Germany,None,"[O, O, O, O, O, O, O, O, O, O, O, B, O, O, O, ..."
2,মার্কিন যুক্তরাষ্ট্রের সংবিধান অনুযায়ী মার্কিন...,Same-sex marriage in the United States expande...,bn,False,-1,no,None,"[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
3,আরব-ইসরায়েলি যুদ্ধে আরবের মোট কয়জন সৈন্যের মৃ...,The exact number of Arab casualties is unknown...,bn,True,39,unknown,None,"[O, O, O, O, O, O, O, O, B, I, I, O, O, O, O, ..."
4,বিশ্বে প্রথম পুঁজিবাদী সমাজ কবে গড়ে ওঠে ?,"As Thomas Hall (2000) notes, ""The Sung Empire ...",bn,True,1219,17th century,None,"[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."


## Automatic checks

In [40]:
def answers_at_0():
  return df_train[
      df_train["answer_start"] == 0
  ]

def multi_token_answers():
  return df_train[
    df_train["sequence_labels"].apply(
        lambda labels: sum(
            label in {"B", "I"} for label in labels
        ) > 1
    )
]

import unicodedata

def is_punctuation(character):
  return (
      character is not None
      and unicodedata.category(character).startswith("P")
  )

def has_adjacent_punctuation(row):
  context = row["context"]
  answer = row["answer"]
  answer_start = row["answer_start"]

  answer_start = int(answer_start)

  if answer_start < 0:
      return False

  answer_end = answer_start + len(answer)

  character_before = (
      context[answer_start - 1]
      if answer_start > 0
      else None
  )

  character_after = (
      context[answer_end]
      if answer_end < len(context)
      else None
  )

  return (
      is_punctuation(character_before)
      or is_punctuation(character_after)
  )

def answers_adjacent_to_punctuation():
  return df_train[
      df_train.apply(
          has_adjacent_punctuation,
          axis=1,
      )
  ]

def unanswerable_answers():
  return df_train[
      df_train["answerable"] == False
  ]


In [37]:
def test_answer_at_0():
  rows = answers_at_0().sample(5)

  for row in rows.itertuples():
    sequence_labels = row.sequence_labels

    assert sequence_labels[0] == "B"

test_answer_at_0()

In [39]:
def test_multi_token_answers():
  rows = multi_token_answers().sample(5)

  for row in rows.itertuples():
    sequence_labels = row.sequence_labels

    assert sequence_labels.count("B") + sequence_labels.count("I") > 1

test_multi_token_answers()

In [44]:
def test_answers_adjacent_to_punctuation():
  rows = answers_adjacent_to_punctuation().sample(5)

  for row in rows.itertuples():
    answer = row.answer
    answer_start = row.answer_start
    answer_end = answer_start + len(answer)
    context = row.context

    char_before = None
    char_after = None
    if answer_start > 0:
      char_before = context[answer_start - 1]

    if answer_end < len(context):
      char_after = context[answer_end]

    assert (
        is_punctuation(char_before)
        or is_punctuation(char_after)
    )

test_answers_adjacent_to_punctuation()


In [46]:
def test_unanswerable_answers():
  rows = unanswerable_answers().sample(5)

  for row in rows.itertuples():
    answer_start = row.answer_start
    sequence_labels = row.sequence_labels

    assert answer_start == -1
    assert sequence_labels.count("B") + sequence_labels.count("I") == 0

test_unanswerable_answers()